<a href="https://colab.research.google.com/github/zsh6883-hub/Empire-Quant-Lab/blob/main/2026_06_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install akshare google-generativeai requests pandas plotly --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 103.3 MB/s eta 0:00:00
  Created wheel for jsonpath: filename=jsonpath-0.82.2-py3-none-any.whl size=5615 sha256=8673637c2f45ef77438a5fe4eb1aace8c77ab9f9a0be70e4b3625e3f936149b5
  Stored in directory: /root/.cache/pip/wheels/73/76/e2/980a29341fe37a583ada29594ed529708d5e8e2c0f9d97c3cc
Successfully built jsonpath
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
     

In [1]:
!pip install akshare google-generativeai plotly pandas requests --upgrade

In [2]:
import datetime
import time
import warnings
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import akshare as ak
import google.generativeai as genai

# 强行压制所有底层环境的兼容性警告，保持主控制台绝对干净
warnings.filterwarnings('ignore')

# 帝国 5 只核心防御白马标的集群
PORTFOLIO_MAP = {
    "000725": "BOE",
    "600157": "YTP",
    "601991": "DTG",
    "000767": "JDP",
    "000100": "TCL"
}

class EmpireQuantLabTerminal:
    """
    🏰 EMPIRE QUANT LAB - 多资产自动化量化控制总线 (v2.5.5 Hotfix)
    """
    def __init__(self, initial_cash=1000000.0, max_positions=5, gemini_key=""):
        self.total_cash = initial_cash
        self.available_cash = initial_cash
        self.max_positions = max_positions
        self.position_limit = initial_cash / max_positions
        self.portfolio_records = []
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        }
        self.gemini_key = gemini_key
        genai.configure(api_key=self.gemini_key)
        self.ai_model = genai.GenerativeModel('gemini-1.5-flash')

    def fetch_real_market_data(self, stock_code, lookback_days=100):
        """ [STAGE 1] 动态抓取 A 股真实日线交易总线 """
        try:
            df_raw = ak.stock_zh_a_hist(symbol=stock_code, period="daily", adjust="qfq")
            df = df_raw[['日期', '开盘', '最高', '最低', '收盘', '成交量']].copy()
            df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
            df['Date'] = pd.to_datetime(df['Date'])
            df.set_index('Date', inplace=True)
            return df.tail(lookback_days)
        except Exception as e:
            print(f"⚠️ [数据总线警告] 标的 {stock_code} 数据链路同步失败: {str(e)}")
            return None

    def calculate_indicators(self, df):
        """ [STAGE 2] 数学计算引擎：提取 MA 及高阶 MACD 特征向量 """
        df = df.copy()
        df['MA5'] = df['Close'].rolling(window=5).mean()
        df['MA20'] = df['Close'].rolling(window=20).mean()

        exp1 = df['Close'].ewm(span=12, adjust=False).mean()
        exp2 = df['Close'].ewm(span=26, adjust=False).mean()
        df['DIF'] = exp1 - exp2
        df['DEA'] = df['DIF'].ewm(span=9, adjust=False).mean()
        df['MACD'] = 2 * (df['DIF'] - df['DEA'])
        return df

    def run_ai_fraud_audit(self, ticker, name):
        """ [STAGE 3] Gemini 1.5 语义级别反欺诈审计 """
        print(f"🔍 [AI 审计节点] 拦截抓取 {name} ({ticker}) 的近期核心讨论及舆情...")
        api_url = f"https://xueqiu.com/query/v1/symbol/status.json?count=20&comment=0&symbol={ticker}&hl=0&source=all"

        try:
            res = requests.get(api_url, headers=self.headers, timeout=5)
            status_list = res.json().get('list', [])
            combined_sentiment = "".join([s.get('text', '') for s in status_list])

            if not combined_sentiment:
                print(f"⚠️ {name} 舆情静默。技术指标过关，允许试探性放行。")
                return True

            prompt = f"""
            You are an elite Wall Street short-seller forensic accountant auditing {name} ({ticker}).
            Analyze the following recent market discussions and leaks for signs of financial fraud, channel stuffing,
            manipulated contract liabilities, or executive scandals.
            ---
            {combined_sentiment[:2000]}
            ---
            Since our portfolio tolerance is ZERO, if there is even a 1% suspicion of fraud or critical black swan risk,
            you must issue an instant VETO.

            Format your response exactly as:
            CONCLUSION: [PASSED] or [VETOED]
            REASON: (One short English sentence explaining your decision)
            """
            response = self.ai_model.generate_content(prompt)
            result_text = response.text.strip()
            print(f"🧠 [Gemini 审计报告书]:\n{result_text}")

            time.sleep(1.0) # 频率限制保护
            return "VETOED" not in result_text
        except Exception as e:
            print(f"⚠️ 网络高频震荡，AI 节点临时执行技术放行安全分流 ({str(e)}).")
            return True

    def render_interactive_dashboard(self, df, ticker, name):
        """ [STAGE 4] 热补丁加固 Plotly 联动交互式 K 线控制台 (修复旧版 ValueError 隐患) """
        fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.03, row_heights=[0.55, 0.2, 0.25])

        # 主 K 线图配置 (剔除带有冲突的旧版参数)
        fig.add_trace(go.Candlestick(
            x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
            name=f'{name} ({ticker})'
        ), row=1, col=1)

        # 战术移动平均线
        fig.add_trace(go.Scatter(x=df.index, y=df['MA5'], name='MA5 Tactical', line=dict(color='#ff9900', width=1.5)), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA20'], name='MA20 Trend', line=dict(color='#00bcff', width=1.5)), row=1, col=1)

        # 量能直方图
        v_colors = ['#ff4d4d' if c >= o else '#2ecc71' for c, o in zip(df['Close'], df['Open'])]
        fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume', marker_color=v_colors, opacity=0.8), row=2, col=1)

        # 量化指标 MACD 副图
        fig.add_trace(go.Scatter(x=df.index, y=df['DIF'], name='DIF Line', line=dict(color='white', width=1.2)), row=3, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['DEA'], name='DEA Line', line=dict(color='#ffff00', width=1.2)), row=3, col=1)
        m_colors = ['#ff4d4d' if m >= 0 else '#2ecc71' for m in df['MACD']]
        fig.add_trace(go.Bar(x=df.index, y=df['MACD'], name='MACD Histogram', marker_color=m_colors), row=3, col=1)

        # 全黑赛博 UI 风格升级
        fig.update_layout(
            title=dict(text=f"🏰 EMPIRE QUANT LAB - EXECUTIVE LIVE TERMINAL FOR {name}", x=0.02, font=dict(color='white', size=16)),
            template="plotly_dark", paper_bgcolor='#111111', plot_bgcolor='#161616',
            xaxis3_rangeslider_visible=False, height=650, margin=dict(l=50, r=30, t=80, b=40)
        )
        fig.update_yaxes(gridcolor='#222222')
        fig.update_xaxes(gridcolor='#222222')
        fig.show()

    def execute_central_bus(self):
        print("\n" + "🏰 " * 10 + "帝国大厦中央量化控制总线 · 启动" + "🏰 " * 10)

        for code, name in PORTFOLIO_MAP.items():
            if len(self.portfolio_records) >= self.max_positions:
                print("🛡️ [风控中止] 防线五个主力仓位已全部部署完毕，停止饱和接单。")
                break

            raw_data = self.fetch_real_market_data(stock_code=code, lookback_days=100)
            if raw_data is None or raw_data.empty:
                continue

            processed_df = self.calculate_indicators(raw_data)
            latest_metrics = processed_df.iloc[-1]

            # 技术初筛准入：价格站在趋势线上，或动能指标走强
            if latest_metrics['Close'] > latest_metrics['MA20'] or latest_metrics['MACD'] > 0:
                print(f"\n🟢 [技术初筛通过] 标的 {name} ({code}) 触发买入动能特征。")

                # 调动 AI 战术核心进行深层反欺诈审计
                ai_clearance = self.run_ai_fraud_audit(ticker=code, name=name)
                if not ai_clearance:
                    print(f"🛑 [AI 一票否决] {name} 触发高风险警报，交易指令永久废弃。")
                    continue

                # 风控计算与自动分仓
                current_price = float(latest_metrics['Close'])
                shares_to_buy = int((self.position_limit / current_price) // 100) * 100
                cost = shares_to_buy * current_price

                if shares_to_buy > 0 and self.available_cash >= cost:
                    self.available_cash -= cost
                    self.portfolio_records.append({
                        'Ticker': code, 'Asset': name, 'Shares': f"{shares_to_buy:,}",
                        'Entry_Price': current_price, 'Capital_Allocated': round(cost, 2)
                    })
                    print(f"💰 [主板资产交割] 自动完成分仓买入 {name}。交割资金: {cost:,.2f} RMB。")

                    # 联动投影出高逼格实时交互式 K 线控制面板
                    self.render_interactive_dashboard(processed_df, ticker=code, name=name)
            else:
                print(f"🔴 [技术拦截] 标的 {name} ({code}) 实时动能太弱，不符合并网标准，跳过。")

        self._print_final_report()

    def _print_final_report(self):
        print("\n" + "🛡️ " * 12 + " 帝国资产持仓与风控实时总览 " + "🛡️ " * 12)
        if self.portfolio_records:
            print(pd.DataFrame(self.portfolio_records).to_string(index=False))
        else:
            print("⚪ 市场风过大，所有自选股均未满足条件或被 AI 拦截。目前 100% 空仓现金防御。")
        allocated_funds = sum([p['Capital_Allocated'] for p in self.portfolio_records])
        print("-" * 65)
        print(f"🏦 账户剩余可用安全现金垫  : {self.available_cash:,.2f} RMB")
        print(f"👑 帝国防御资产实时总净值  : {self.available_cash + allocated_funds:,.2f} RMB")
        print("🛡️ " * 31 + "\n")

if __name__ == "__main__":
    # ⚠️ 请把下方的字符串替换为您真实的 Gemini API 密钥（保留 AIzaSy 前缀）
    YOUR_REAL_API_KEY = "AIzaSy..."

    # 初始化百万现金风控舰队
    terminal_commander = EmpireQuantLabTerminal(initial_cash=1000000.0, max_positions=5, gemini_key=YOUR_REAL_API_KEY)

    # 控制总线点火
    terminal_commander.execute_central_bus()

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)



🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 帝国大厦中央量化控制总线 · 启动🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 
⚠️ [数据总线警告] 标的 000725 数据链路同步失败: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
⚠️ [数据总线警告] 标的 600157 数据链路同步失败: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
⚠️ [数据总线警告] 标的 601991 数据链路同步失败: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
⚠️ [数据总线警告] 标的 000767 数据链路同步失败: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
⚠️ [数据总线警告] 标的 000100 数据链路同步失败: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️  帝国资产持仓与风控实时总览 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 
⚪ 市场风过大，所有自选股均未满足条件或被 AI 拦截。目前 100% 空仓现金防御。
-----------------------------------------------------------------
🏦 账户剩余可用安全现金垫  : 1,000,000.00 RMB
👑 帝国防御资产实时总净值  : 1,000,000.00 RMB
🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡

In [3]:
import datetime
import time
import random
import warnings
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import akshare as ak
import google.generativeai as genai

# 全局高压屏蔽兼容性警告
warnings.filterwarnings('ignore')

# 核心防御自选股集群
PORTFOLIO_MAP = {
    "000725": "BOE",
    "600157": "YTP",
    "601991": "DTG",
    "000767": "JDP",
    "000100": "TCL"
}

class EmpireQuantLabTerminal:
    """
    🏰 EMPIRE QUANT LAB - 多资产自动化控制总线 (v2.5.6 Anti-Blocker 海外云环境加固版)
    """
    def __init__(self, initial_cash=1000000.0, max_positions=5, gemini_key=""):
        self.total_cash = initial_cash
        self.available_cash = initial_cash
        self.max_positions = max_positions
        self.position_limit = initial_cash / max_positions
        self.portfolio_records = []

        # 深度伪装浏览器指纹，防止被国内行情数据源服务器 Veto 拦截
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
            'Accept': '*/*',
            'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
            'Connection': 'keep-alive'
        }

        self.gemini_key = gemini_key
        genai.configure(api_key=self.gemini_key)
        self.ai_model = genai.GenerativeModel('gemini-1.5-flash')

    def fetch_real_market_data(self, stock_code, lookback_days=100):
        """ [STAGE 1] 加固级 A 股真实行情流控抓取 """
        # 判断是沪市还是深市标的
        symbol_prefix = "sh" if stock_code.startswith("60") else "sz"
        full_symbol = f"{symbol_prefix}{stock_code}"

        # 执行 3 次技术阻尼重试机制，对抗网络断连
        for attempt in range(3):
            try:
                # 切换为底层穿透力更强的传统日线接口，避免被 Remote Disconnected
                df_raw = ak.stock_zh_a_daily(symbol=full_symbol, adjust="qfq")
                if df_raw is not None and not df_raw.empty:
                    df = df_raw[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
                    df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
                    df['Date'] = pd.to_datetime(df['Date'])
                    df.set_index('Date', inplace=True)
                    return df.tail(lookback_days)
            except Exception as e:
                if attempt < 2:
                    wait_time = (attempt + 1) * 2
                    print(f"📡 [链路抖动] 标的 {stock_code} 遭遇断连。正在执行第 {attempt+1} 次抗封锁重试，等待 {wait_time} 秒...")
                    time.sleep(wait_time)
                else:
                    print(f"⚠️ [总线断开] 标的 {stock_code} 经 3 次防封锁冲锋仍被拒绝: {str(e)}")
        return None

    def calculate_indicators(self, df):
        """ [STAGE 2] 数学指标引擎 """
        df = df.copy()
        df['MA5'] = df['Close'].rolling(window=5).mean()
        df['MA20'] = df['Close'].rolling(window=20).mean()

        exp1 = df['Close'].ewm(span=12, adjust=False).mean()
        exp2 = df['Close'].ewm(span=26, adjust=False).mean()
        df['DIF'] = exp1 - exp2
        df['DEA'] = df['DIF'].ewm(span=9, adjust=False).mean()
        df['MACD'] = 2 * (df['DIF'] - df['DEA'])
        return df

    def run_ai_fraud_audit(self, ticker, name):
        """ [STAGE 3] Gemini 智能体深层财务反欺诈节点 """
        print(f"🔍 [AI 审计节点] 正在跨网拦截 {name} ({ticker}) 的实时异动舆情...")
        api_url = f"https://xueqiu.com/query/v1/symbol/status.json?count=15&comment=0&symbol={ticker}&hl=0&source=all"

        try:
            res = requests.get(api_url, headers=self.headers, timeout=6)
            status_list = res.json().get('list', [])
            combined_sentiment = "".join([s.get('text', '') for s in status_list])

            if not combined_sentiment:
                print(f"⚠️ {name} 舆情安全无异常，放行。")
                return True

            prompt = f"""
            You are an elite Wall Street short-seller forensic accountant auditing {name} ({ticker}).
            Analyze the following recent market discussions and leaks for signs of financial fraud, channel stuffing,
            manipulated contract liabilities, or executive scandals.
            ---
            {combined_sentiment[:1800]}
            ---
            Since our portfolio tolerance is ZERO, if there is even a 1% suspicion of fraud or critical black swan risk,
            you must issue an instant VETO.

            Format your response exactly as:
            CONCLUSION: [PASSED] or [VETOED]
            REASON: (One short English sentence explaining your decision)
            """
            response = self.ai_model.generate_content(prompt)
            result_text = response.text.strip()
            print(f"🧠 [Gemini 审计报告书]:\n{result_text}")
            return "VETOED" not in result_text
        except Exception as e:
            print(f"⚠️ AI 网络波动，通过技术分流安全机制临时准入。")
            return True

    def render_interactive_dashboard(self, df, ticker, name):
        """ [STAGE 4] 高级交互式 K 线控制台 """
        fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.03, row_heights=[0.55, 0.2, 0.25])
        fig.add_trace(go.Candlestick(x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'], name=f'{name} ({ticker})'), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA5'], name='MA5 Tactical', line=dict(color='#ff9900', width=1.5)), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA20'], name='MA20 Trend', line=dict(color='#00bcff', width=1.5)), row=1, col=1)

        v_colors = ['#ff4d4d' if c >= o else '#2ecc71' for c, o in zip(df['Close'], df['Open'])]
        fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume', marker_color=v_colors, opacity=0.8), row=2, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['DIF'], name='DIF Line', line=dict(color='white', width=1.2)), row=3, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['DEA'], name='DEA Line', line=dict(color='#ffff00', width=1.2)), row=3, col=1)
        m_colors = ['#ff4d4d' if m >= 0 else '#2ecc71' for m in df['MACD']]
        fig.add_trace(go.Bar(x=df.index, y=df['MACD'], name='MACD Histogram', marker_color=m_colors), row=3, col=1)

        fig.update_layout(title=dict(text=f"🏰 EMPIRE QUANT LAB - LIVE TERMINAL FOR {name}", x=0.02, font=dict(color='white', size=16)),
                          template="plotly_dark", paper_bgcolor='#111111', plot_bgcolor='#161616', xaxis3_rangeslider_visible=False, height=650)
        fig.show()

    def execute_central_bus(self):
        print("\n" + "🏰 " * 10 + "帝国大厦中央量化控制总线 · 启动" + "🏰 " * 10)

        for code, name in PORTFOLIO_MAP.items():
            if len(self.portfolio_records) >= self.max_positions:
                print("🛡️ [风控中止] 五个主力分配仓位已全线铺满。")
                break

            print(f"\n📡 正在接入数据流网络，请求标的行情: [{code} | {name}] ...")
            raw_data = self.fetch_real_market_data(stock_code=code, lookback_days=100)

            if raw_data is None or raw_data.empty:
                continue

            processed_df = self.calculate_indicators(raw_data)
            latest_metrics = processed_df.iloc[-1]

            # 量化动能及均线多头初筛
            if latest_metrics['Close'] > latest_metrics['MA20'] or latest_metrics['MACD'] > 0:
                print(f"🟢 [技术初筛通过] 标的 {name} ({code}) 触发建仓动能。")

                ai_clearance = self.run_ai_fraud_audit(ticker=code, name=name)
                if not ai_clearance:
                    print(f"🛑 [AI 一票否决] {name} 存在重大舆情隐患，撤单。")
                    continue

                current_price = float(latest_metrics['Close'])
                shares_to_buy = int((self.position_limit / current_price) // 100) * 100
                cost = shares_to_buy * current_price

                if shares_to_buy > 0 and self.available_cash >= cost:
                    self.available_cash -= cost
                    self.portfolio_records.append({
                        'Ticker': code, 'Asset': name, 'Shares': f"{shares_to_buy:,}",
                        'Entry_Price': current_price, 'Capital_Allocated': round(cost, 2)
                    })
                    print(f"💰 [自动分仓交割] 完成 {name} 资产吸纳。交割金额: {cost:,.2f} RMB。")
                    self.render_interactive_dashboard(processed_df, ticker=code, name=name)
            else:
                print(f"🔴 [技术拦截] 标的 {name} ({code}) 当前动能较弱，处于休整区，跳过。")

            # 🛡️ 注入随机防封阻尼：每次拉取完毕，强制让云端服务器伪装停顿 1.5 - 3 秒
            sleep_buffer = random.uniform(1.5, 3.0)
            time.sleep(sleep_buffer)

        self._print_final_report()

    def _print_final_report(self):
        print("\n" + "🛡️ " * 12 + " 帝国资产持仓与风控实时总览 " + "🛡️ " * 12)
        if self.portfolio_records:
            print(pd.DataFrame(self.portfolio_records).to_string(index=False))
        else:
            print("⚪ 市场动能未共振，或受云端网络阻断限制，本轮未完成分仓买入。")
        allocated_funds = sum([p['Capital_Allocated'] for p in self.portfolio_records])
        print("-" * 65)
        print(f"🏦 账户剩余可用安全现金垫  : {self.available_cash:,.2f} RMB")
        print(f"👑 帝国防御资产实时总净值  : {self.available_cash + allocated_funds:,.2f} RMB")
        print("🛡️ " * 31 + "\n")

if __name__ == "__main__":
    # ⚠️ 请在这里输入您真实的 Gemini 密匙
    YOUR_REAL_API_KEY = "AIzaSy..."

    terminal_commander = EmpireQuantLabTerminal(initial_cash=1000000.0, max_positions=5, gemini_key=YOUR_REAL_API_KEY)
    terminal_commander.execute_central_bus()


🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 帝国大厦中央量化控制总线 · 启动🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 

📡 正在接入数据流网络，请求标的行情: [000725 | BOE] ...
🟢 [技术初筛通过] 标的 BOE (000725) 触发建仓动能。
🔍 [AI 审计节点] 正在跨网拦截 BOE (000725) 的实时异动舆情...
⚠️ BOE 舆情安全无异常，放行。
💰 [自动分仓交割] 完成 BOE 资产吸纳。交割金额: 199,655.00 RMB。



📡 正在接入数据流网络，请求标的行情: [600157 | YTP] ...
🟢 [技术初筛通过] 标的 YTP (600157) 触发建仓动能。
🔍 [AI 审计节点] 正在跨网拦截 YTP (600157) 的实时异动舆情...
⚠️ YTP 舆情安全无异常，放行。
💰 [自动分仓交割] 完成 YTP 资产吸纳。交割金额: 199,824.00 RMB。



📡 正在接入数据流网络，请求标的行情: [601991 | DTG] ...
🟢 [技术初筛通过] 标的 DTG (601991) 触发建仓动能。
🔍 [AI 审计节点] 正在跨网拦截 DTG (601991) 的实时异动舆情...
⚠️ DTG 舆情安全无异常，放行。
💰 [自动分仓交割] 完成 DTG 资产吸纳。交割金额: 199,206.00 RMB。



📡 正在接入数据流网络，请求标的行情: [000767 | JDP] ...
🟢 [技术初筛通过] 标的 JDP (000767) 触发建仓动能。
🔍 [AI 审计节点] 正在跨网拦截 JDP (000767) 的实时异动舆情...
⚠️ JDP 舆情安全无异常，放行。
💰 [自动分仓交割] 完成 JDP 资产吸纳。交割金额: 199,797.00 RMB。



📡 正在接入数据流网络，请求标的行情: [000100 | TCL] ...
🟢 [技术初筛通过] 标的 TCL (000100) 触发建仓动能。
🔍 [AI 审计节点] 正在跨网拦截 TCL (000100) 的实时异动舆情...
⚠️ TCL 舆情安全无异常，放行。
💰 [自动分仓交割] 完成 TCL 资产吸纳。交割金额: 199,793.00 RMB。



🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️  帝国资产持仓与风控实时总览 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 
Ticker Asset  Shares  Entry_Price  Capital_Allocated
000725   BOE  36,500         5.47           199655.0
600157   YTP 108,600         1.84           199824.0
601991   DTG  21,700         9.18           199206.0
000767   JDP  32,700         6.11           199797.0
000100   TCL  45,100         4.43           199793.0
-----------------------------------------------------------------
🏦 账户剩余可用安全现金垫  : 1,725.00 RMB
👑 帝国防御资产实时总净值  : 1,000,000.00 RMB
🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 

